## tl;dr

From 2025-01-01 through 20260924, the v2.24 state machine produced 6,473 rows across both simulations. For deduplicated v2.24 events with a mature 10-session outcome, next-open hit rate was 47.14%, mean return was 0.69%, and median return was -0.52%. The v2.24 minus counterfactual 10-session mean-return difference was -0.06%; its bootstrap interval crossed zero. The rule increased signal volume but did not establish a statistically reliable quality improvement.

## Context & Methods

### Key Assumptions

- The canonical simulation applies v2.24's non-retroactive L1/L2 rule.
- The comparison changes only that rule and is a counterfactual, not a complete reconstruction of v2.23.
- Tushare current THS industry membership is applied historically because the interface does not provide effective dates.
- Signal-close returns are diagnostics; next-open returns are the more executable approximation.
- L7 is evaluated because L10 sell and holding rules remain undefined.

In [1]:
from pathlib import Path
import json
import pandas as pd

output_dir = Path('studies/v224_condition_validation/outputs')
summary = pd.read_csv(output_dir / 'summary.csv')
comparison = pd.read_csv(output_dir / 'comparison.csv')
yearly = pd.read_csv(output_dir / 'yearly.csv')
events = pd.read_parquet(output_dir / 'events.parquet')
qa = json.loads((output_dir / 'qa.json').read_text())

## Data

In [2]:
pd.Series({
    'analysis_start_date': qa['analysis_start_date'],
    'analysis_end_date': qa['analysis_end_date'],
    'panel_rows': qa['panel_rows'],
    'panel_security_count': qa['panel_security_count'],
    'minimum_monthly_daily_basic_complete_rate': qa['minimum_monthly_daily_basic_complete_rate'],
    'minimum_monthly_industry_mapping_rate': qa['minimum_monthly_industry_mapping_rate'],
    'duplicate_panel_keys': qa['duplicate_panel_keys'],
    'duplicate_event_keys': qa['event_duplicate_keys'],
})

analysis_start_date                          20250101
analysis_end_date                            20260924
panel_rows                                    2171772
panel_security_count                             5271
minimum_monthly_daily_basic_complete_rate         1.0
minimum_monthly_industry_mapping_rate        0.993536
duplicate_panel_keys                                0
duplicate_event_keys                                0
dtype: object

In [3]:
pd.DataFrame(qa['spot_checks']).T

,ts_code,trade_date,qf_signal_close,qf_future_close_10,reported_return_10,recalculated_return_10,absolute_difference,amount,prior_five_day_mean_amount,reported_ratio,recalculated_ratio,distance_from_threshold
positive_return,002184.SZ,20250102,42.529645,49.844082,0.171984,0.171984,0.0,NaN,NaN,NaN,NaN,NaN
negative_return,000417.SZ,20250102,131.06691,110.90277,-0.153846,-0.153846,0.0,NaN,NaN,NaN,NaN,NaN
amount_boundary,600377.SH,20250519,NaN,NaN,NaN,NaN,NaN,146290.484,97525.8828,1.500017,1.500017,0.000017


## Results

In [4]:
summary[(summary['sample_type'] == 'keep_first_10d') & summary['horizon'].isin([1, 3, 5, 10])][[
    'group', 'entry_basis', 'horizon', 'event_count', 'hit_rate',
    'mean_return', 'median_return', 'trimmed_mean_5pct',
    'mean_excess_return', 'mean_return_ci_low', 'mean_return_ci_high'
]]

,group,entry_basis,horizon,event_count,hit_rate,mean_return,median_return,trimmed_mean_5pct,mean_excess_return,mean_return_ci_low,mean_return_ci_high
20,v2.24,signal_close,1,5069,0.438548,0.000348,-0.002768,-0.001026,0.000049,-0.001449,0.002112
22,v2.24,signal_close,3,5040,0.457937,0.001682,-0.003483,-0.001535,-0.000270,-0.001809,0.005087
24,v2.24,signal_close,5,4962,0.463926,0.002283,-0.004686,-0.002209,-0.000979,-0.001502,0.006032
29,v2.24,signal_close,10,4949,0.466761,0.006420,-0.005457,-0.000993,-0.002920,0.000411,0.012583
30,v2.24,next_open,1,5069,0.461432,0.000933,-0.001549,-0.000111,-0.000027,-0.000715,0.002646
32,v2.24,next_open,3,5040,0.471825,0.002128,-0.002182,-0.000445,-0.000453,-0.001194,0.005502
34,v2.24,next_open,5,4962,0.470173,0.002707,-0.003696,-0.001362,-0.001239,-0.001018,0.006424
39,v2.24,next_open,10,4949,0.471408,0.006881,-0.005230,-0.000229,-0.003158,0.001086,0.013061
60,retroactive_counterfactual,signal_close,1,1395,0.427957,-0.000282,-0.004392,-0.001639,-0.000933,-0.002980,0.002474
62,retroactive_counterfactual,signal_close,3,1385,0.454874,0.002937,-0.004630,-0.000348,0.000188,-0.002163,0.007269


In [5]:
comparison[comparison['horizon'].isin([1, 3, 5, 10])][[
    'comparison', 'entry_basis', 'horizon', 'hit_rate_difference',
    'mean_return_difference', 'hit_rate_difference_ci_low',
    'hit_rate_difference_ci_high', 'mean_return_difference_ci_low',
    'mean_return_difference_ci_high'
]]

,comparison,entry_basis,horizon,hit_rate_difference,mean_return_difference,hit_rate_difference_ci_low,hit_rate_difference_ci_high,mean_return_difference_ci_low,mean_return_difference_ci_high
0,v2.24_minus_counterfactual,signal_close,1,0.010591,0.000630,-0.031652,0.050999,-0.002738,0.003824
2,v2.24_minus_counterfactual,signal_close,3,0.003063,-0.001255,-0.038029,0.045048,-0.006983,0.004270
4,v2.24_minus_counterfactual,signal_close,5,0.010641,0.001339,-0.030713,0.051137,-0.006660,0.009455
9,v2.24_minus_counterfactual,signal_close,10,0.013613,0.000489,-0.032655,0.057359,-0.010728,0.012162
10,v2.24_minus_counterfactual,next_open,1,0.000500,-0.000520,-0.046020,0.039783,-0.003889,0.002348
12,v2.24_minus_counterfactual,next_open,3,-0.006875,-0.002303,-0.048409,0.036997,-0.007460,0.003308
14,v2.24_minus_counterfactual,next_open,5,0.005210,0.000306,-0.039473,0.046522,-0.007392,0.007915
19,v2.24_minus_counterfactual,next_open,10,0.006547,-0.000550,-0.040289,0.050506,-0.011310,0.010899
20,incremental_minus_counterfactual,signal_close,1,0.014799,0.000872,-0.026765,0.056159,-0.002176,0.004043
22,incremental_minus_counterfactual,signal_close,3,0.004357,-0.001719,-0.038425,0.048395,-0.007647,0.004266


In [6]:
yearly

,version,signal_year,horizon,event_count,hit_rate,mean_return,median_return
0,retroactive_counterfactual,2025,5,938,0.452026,0.002766,-0.007252
1,retroactive_counterfactual,2025,10,938,0.464819,0.012469,-0.006541
2,retroactive_counterfactual,2026,5,432,0.456019,-0.003013,-0.008190
3,retroactive_counterfactual,2026,10,428,0.427570,-0.008395,-0.021534
4,v2.24,2025,5,3576,0.466723,0.003420,-0.003661
5,v2.24,2025,10,3576,0.486018,0.010470,-0.001725
6,v2.24,2026,5,1386,0.456710,-0.000651,-0.008701
7,v2.24,2026,10,1373,0.416606,-0.004126,-0.019362


## Takeaways

1. V2.24 materially expands the number of L7 signals.
2. Ten-session mean returns are positive, but hit rates and medians are below zero-centered thresholds, indicating a right-tail payoff profile.
3. Mean excess returns are negative against the same-date equal-weight SH/SZ benchmark.
4. The v2.24 versus counterfactual differences have confidence intervals that cross zero; predictive improvement is not established.
5. The historical industry-map limitation prevents a fully point-in-time production acceptance.